# 🎨 ComfyUI Google Colab — Bản Nâng Cấp Đầy Đủ Tính Năng

| Tính năng | Trạng thái |
|-----------|------------|
| 🖥️ Thông tin GPU & máy chủ | ✅ Tự động |
| 🔑 Nhập Civitai Token an toàn | ✅ Form nhập |
| 📦 Cài ComfyUI + Nodes + Models | ✅ Tự động |
| 💾 Lưu trữ vĩnh viễn trên Google Drive | ✅ Symlink |
| 🔧 Tự động sửa lỗi thiếu package | ✅ fix_nodes |
| 🌐 3 đường hầm truy cập (Colab + CF + LT) | ✅ Đa dự phòng |
| 📥 Tải thêm Model/LoRA/Node bất kỳ | ✅ Cell riêng |
| 🔄 Khởi động lại không mất dữ liệu | ✅ Reload cell |
| 📊 Xem báo cáo tổng kết cài đặt | ✅ Cell riêng |

**👉 Hướng dẫn**: Chọn **Runtime → Run all** (`Ctrl+F9`) để chạy từ A đến Z tự động!

In [ ]:
# @title ⚙️ Bước 0: Cấu Hình — Nhập Token & Tuỳ Chọn
# @markdown ---
# @markdown ### 🔑 Thông tin bắt buộc
GITHUB_REPO_URL = "https://github.com/hung187/comfyui-setup1.git" # @param {type:"string"}
CIVITAI_TOKEN = "" # @param {type:"string"}
# @markdown ---
# @markdown ### ⚙️ Tuỳ chọn cài đặt
CHI_CAI_NODES = False # @param {type:"boolean"}
CHI_TAI_MODELS = False # @param {type:"boolean"}
TU_DONG_SUA_LOI_NODE = True # @param {type:"boolean"}
# @markdown ---
# @markdown ### 🔗 Tuỳ chọn Tunnel
BAT_CLOUDFLARE = True # @param {type:"boolean"}
BAT_LOCALTUNNEL = True # @param {type:"boolean"}

import os
# Validate token
if not CIVITAI_TOKEN.strip():
    print("⚠️  CIVITAI_TOKEN trống! Các model từ Civitai sẽ không tải được.")
    print("   → Lấy token tại: https://civitai.com/user/account → API Keys")
else:
    masked = CIVITAI_TOKEN[:4] + "****" + CIVITAI_TOKEN[-4:]
    print(f"✅ Civitai Token đã được nhận: {masked}")

print("\n📋 Cấu hình hiện tại:")
print(f"   Repo    : {GITHUB_REPO_URL}")
print(f"   Chỉ Nodes  : {CHI_CAI_NODES}")
print(f"   Chỉ Models : {CHI_TAI_MODELS}")
print(f"   Sửa lỗi Node: {TU_DONG_SUA_LOI_NODE}")
print(f"   Cloudflare : {BAT_CLOUDFLARE}")
print(f"   Localtunnel: {BAT_LOCALTUNNEL}")

In [ ]:
# @title 🖥️ Bước 1: Kiểm Tra GPU & Thông Tin Máy Chủ
import subprocess, platform, os

print("╔══════════════════════════════════════════════════════╗")
print("║        🖥️  THÔNG TIN MÁY CHỦ GOOGLE COLAB           ║")
print("╚══════════════════════════════════════════════════════╝")

# GPU
print("\n🎮 GPU:")
result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.free,temperature.gpu",
     "--format=csv,noheader"],
    capture_output=True, text=True
)
if result.returncode == 0:
    for line in result.stdout.strip().split("\n"):
        parts = [p.strip() for p in line.split(",")]
        if len(parts) >= 3:
            print(f"   Tên: {parts[0]}")
            print(f"   VRAM Tổng: {parts[1]} | Trống: {parts[2]}")
            if len(parts) >= 4:
                print(f"   Nhiệt độ: {parts[3]}")
else:
    print("   ⚠️  Không tìm thấy GPU! Vào Runtime → Change runtime type → GPU")

# RAM
print("\n🧠 RAM:")
with open("/proc/meminfo") as f:
    mem = {}
    for line in f:
        k, v = line.split(":", 1)
        mem[k.strip()] = int(v.strip().split()[0])
total_gb = mem["MemTotal"] / 1024 / 1024
avail_gb = mem["MemAvailable"] / 1024 / 1024
print(f"   Tổng: {total_gb:.1f} GB | Trống: {avail_gb:.1f} GB")

# Disk
print("\n💾 Ổ Đĩa:")
result = subprocess.run(["df", "-BG", "/"], capture_output=True, text=True)
lines = result.stdout.strip().split("\n")
if len(lines) >= 2:
    parts = lines[1].split()
    total_d = parts[1].replace("G","")
    used_d  = parts[2].replace("G","")
    free_d  = parts[3].replace("G","")
    warn = " ⚠️ THẤP!" if int(free_d) < 20 else ""
    print(f"   Tổng: {total_d}GB | Dùng: {used_d}GB | Trống: {free_d}GB{warn}")

# Python
print(f"\n🐍 Python: {platform.python_version()}")

# IP
import urllib.request
try:
    pub_ip = urllib.request.urlopen("https://api.ipify.org", timeout=5).read().decode()
    print(f"🌐 IP Công Khai: {pub_ip}")
except:
    print("🌐 IP Công Khai: (không lấy được)")

print("\n══════════════════════════════════════════════════════")

In [ ]:
# @title 📁 Bước 2: Gắn Google Drive & Tạo Cấu Trúc Lưu Trữ Vĩnh Viễn
import os
from google.colab import drive

print("📁 Đang kết nối Google Drive...")
drive.mount("/content/drive")

# Cấu trúc thư mục trên Drive
DRIVE_COMFY   = "/content/drive/MyDrive/ComfyUI"
DRIVE_MODELS  = f"{DRIVE_COMFY}/models"
DRIVE_NODES   = f"{DRIVE_COMFY}/custom_nodes"
DRIVE_OUTPUTS = "/content/drive/MyDrive/ComfyUI_Outputs"

for d in [DRIVE_MODELS, DRIVE_NODES, DRIVE_OUTPUTS]:
    os.makedirs(d, exist_ok=True)

print(f"\n✅ Thư mục Models trên Drive  : {DRIVE_MODELS}")
print(f"✅ Thư mục Nodes trên Drive   : {DRIVE_NODES}")
print(f"✅ Thư mục Ảnh Output trên Drive: {DRIVE_OUTPUTS}")
print("\n🎉 Google Drive đã sẵn sàng!")

In [ ]:
# @title 🚀 Bước 3: Cài Đặt ComfyUI + Custom Nodes + Models
import os, subprocess

SETUP_DIR   = "/content/comfyui-setup1"
COMFY_DIR   = "/content/ComfyUI"
TOKEN_CLEAN = CIVITAI_TOKEN.strip()

# 1. Clone hoặc cập nhật bộ script từ GitHub
if not os.path.exists(SETUP_DIR):
    print("📦 Đang tải bộ script setup từ GitHub...")
    !git clone -q "{GITHUB_REPO_URL}" "{SETUP_DIR}"
else:
    print("🔄 Cập nhật bộ script setup từ GitHub...")
    !git -C "{SETUP_DIR}" pull -q

%cd "{SETUP_DIR}"

# 2. Xây dựng câu lệnh install
install_flags = f"--comfy-dir {COMFY_DIR}"
if TOKEN_CLEAN:
    install_flags += f" --civitai-token {TOKEN_CLEAN}"
if CHI_CAI_NODES:
    install_flags += " --only-nodes"
elif CHI_TAI_MODELS:
    install_flags += " --only-models"

print(f"\n🚀 Bắt đầu cài đặt: bash install.sh {install_flags}")
!bash install.sh {install_flags}

# 3. Tạo Symlink liên kết ComfyUI local với Google Drive
print("\n🔗 Đang tạo Symlink liên kết ComfyUI ↔ Google Drive...")

def make_symlink(src_local, dst_drive):
    os.makedirs(dst_drive, exist_ok=True)
    if os.path.exists(src_local) and not os.path.islink(src_local):
        !cp -rn "{src_local}/"* "{dst_drive}/" 2>/dev/null || true
        !rm -rf "{src_local}"
    if not os.path.exists(src_local):
        os.symlink(dst_drive, src_local)
        print(f"   ✅ Linked: {src_local} → {dst_drive}")
    else:
        print(f"   ⏩ Đã có: {src_local}")

DRIVE_COMFY   = "/content/drive/MyDrive/ComfyUI"
DRIVE_MODELS  = f"{DRIVE_COMFY}/models"
DRIVE_NODES   = f"{DRIVE_COMFY}/custom_nodes"
DRIVE_OUTPUTS = "/content/drive/MyDrive/ComfyUI_Outputs"

make_symlink(f"{COMFY_DIR}/models",       DRIVE_MODELS)
make_symlink(f"{COMFY_DIR}/custom_nodes", DRIVE_NODES)
make_symlink(f"{COMFY_DIR}/output",       DRIVE_OUTPUTS)

print("\n🎉 ĐÃ HOÀN TẤT! ComfyUI sẵn sàng chạy!")

In [ ]:
# @title 🔧 Bước 4: Tự Động Sửa Lỗi Thiếu Package (fix_nodes.sh)
# @markdown Chạy cell này nếu có Custom Node bị đỏ lỗi trong ComfyUI sau khi cài.
import os

SETUP_DIR = "/content/comfyui-setup1"

if TU_DONG_SUA_LOI_NODE:
    if os.path.exists(f"{SETUP_DIR}/fix_nodes.sh"):
        print("🔧 Đang tự động sửa lỗi thiếu package cho tất cả Custom Nodes...")
        %cd "{SETUP_DIR}"
        !bash fix_nodes.sh
    else:
        print("⚠️  Không tìm thấy fix_nodes.sh. Hãy chạy Bước 3 trước.")
else:
    print("ℹ️  Tắt tự động sửa lỗi (TU_DONG_SUA_LOI_NODE = False). Bỏ qua bước này.")

In [ ]:
# @title 📥 (Tùy Chọn) Tải Thêm Model / LoRA / Custom Node Bất Kỳ
# @markdown Nhập thông tin để tải thêm tệp bất kỳ (để trống URL nếu không cần):
URL_TAI_THEM  = "" # @param {type:"string"}
LOAI_TAI      = "checkpoints" # @param ["checkpoints", "loras", "controlnet", "vae", "upscale_models", "custom_nodes"]
SUBFOLDER_LORA = "Character" # @param ["Quality", "Face", "Hands", "Character", "Style", "Effect", "Pose"]
TEN_FILE_LUU  = "" # @param {type:"string"}

import os, urllib.request

DRIVE_COMFY = "/content/drive/MyDrive/ComfyUI"
TOKEN_CLEAN = CIVITAI_TOKEN.strip()

if URL_TAI_THEM.strip():
    url = URL_TAI_THEM.strip()

    # Đính kèm token Civitai nếu cần
    if "civitai.com" in url and "token=" not in url and TOKEN_CLEAN:
        sep = "&" if "?" in url else "?"
        url += f"{sep}token={TOKEN_CLEAN}"

    if LOAI_TAI == "custom_nodes":
        target_dir = f"{DRIVE_COMFY}/custom_nodes"
        os.makedirs(target_dir, exist_ok=True)
        node_name = os.path.basename(url.rstrip("/").replace(".git", ""))
        print(f"📦 Đang clone Custom Node: {node_name}")
        !git clone -q "{url}" "{target_dir}/{node_name}"
        print(f"✅ Đã clone xong vào Drive: {target_dir}/{node_name}")
    else:
        # Xác định thư mục đích
        if LOAI_TAI == "loras":
            target_dir = f"{DRIVE_COMFY}/models/loras/{SUBFOLDER_LORA}"
        else:
            target_dir = f"{DRIVE_COMFY}/models/{LOAI_TAI}"
        os.makedirs(target_dir, exist_ok=True)

        # Xác định tên file
        out_name = TEN_FILE_LUU.strip() or os.path.basename(url.split("?")[0])
        out_path = os.path.join(target_dir, out_name)

        print(f"⬇️  Đang tải: {out_name}")
        print(f"   Lưu vào  : {out_path}")

        !aria2c --console-log-level=error --summary-interval=0 \
                -c -x 4 -s 4 -k 1M "{url}" \
                -d "{target_dir}" -o "{out_name}" || \
         wget -c -q --show-progress -O "{out_path}" "{url}" || \
         curl -L -C - -o "{out_path}" "{url}"

        if os.path.exists(out_path) and os.path.getsize(out_path) > 1024*1024:
            size_mb = os.path.getsize(out_path) / 1024 / 1024
            print(f"✅ Tải thành công! ({size_mb:.1f} MB) → {out_path}")
        else:
            print(f"❌ Tải thất bại hoặc file quá nhỏ. Kiểm tra lại URL/Token.")
else:
    print("ℹ️  Ô URL trống, bỏ qua tải thêm.")

In [ ]:
# @title 🌐 Bước 5: Khởi Động ComfyUI & Hiển Thị Link Truy Cập
import os, time, subprocess, re, urllib.request

COMFY_DIR  = "/content/ComfyUI"
COMFY_LOG  = "/content/comfyui.log"
CF_LOG     = "/content/cloudflared.log"
LT_LOG     = "/content/localtunnel.log"

# 1. Cài Cloudflared nếu chưa có
if BAT_CLOUDFLARE and not os.path.exists("/usr/local/bin/cloudflared"):
    print("📦 Cài đặt Cloudflared...")
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
    !rm -f cloudflared-linux-amd64.deb
    print("✅ Cloudflared đã cài xong!")

# 2. Cài Localtunnel nếu cần
if BAT_LOCALTUNNEL:
    !apt-get install -y -qq nodejs npm > /dev/null 2>&1
    !npm install -g localtunnel > /dev/null 2>&1

# 3. Dọn log cũ
for log in [COMFY_LOG, CF_LOG, LT_LOG]:
    if os.path.exists(log):
        os.remove(log)

# 4. Khởi chạy ComfyUI
%cd "{COMFY_DIR}"
comfy_cmd = f"python main.py --listen 0.0.0.0 --port 8188 --enable-cors-header > {COMFY_LOG} 2>&1"
subprocess.Popen(comfy_cmd, shell=True)
print("⏳ Đang khởi động ComfyUI Server...")

# 5. Polling chờ ComfyUI sẵn sàng (tối đa 120 giây)
comfy_ready = False
for i in range(60):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/", timeout=2) as resp:
            if resp.status == 200:
                comfy_ready = True
                break
    except:
        pass
    time.sleep(2)
    if i % 5 == 0:
        print(f"   Đang chờ... ({i*2}s)")

if comfy_ready:
    print("✅ ComfyUI đã sẵn sàng tại 127.0.0.1:8188!")
else:
    print("⚠️  ComfyUI chưa sẵn sàng! Xem log lỗi:")
    if os.path.exists(COMFY_LOG):
        with open(COMFY_LOG) as f:
            content = f.read()
            # Tìm và hiển thị các dòng lỗi
            errors = [l for l in content.split("\n") if "error" in l.lower() or "traceback" in l.lower()]
            print("\n".join(errors[-10:]) if errors else content[-1500:])

# 6. Lấy Google Colab Native Proxy URL (tương thích Python 3.12)
google_proxy_url = None
try:
    from google.colab.output import eval_js
    google_proxy_url = eval_js("google.colab.kernel.proxyPort(8188)")
except Exception:
    pass

# 7. Khởi chạy các tunnel
if BAT_CLOUDFLARE:
    subprocess.Popen(f"cloudflared tunnel --url http://127.0.0.1:8188 > {CF_LOG} 2>&1", shell=True)

if BAT_LOCALTUNNEL:
    subprocess.Popen(f"npx localtunnel --port 8188 > {LT_LOG} 2>&1", shell=True)

# 8. Lấy IP Colab cho Localtunnel password
try:
    colab_ip = urllib.request.urlopen("https://ipv4.icanhazip.com", timeout=5).read().decode().strip()
except:
    colab_ip = "N/A"

# 9. Chờ và quét log lấy URL tunnel
cf_url = None
lt_url = None
for _ in range(30):
    time.sleep(1)
    if not cf_url and os.path.exists(CF_LOG):
        with open(CF_LOG) as f:
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
            if m:
                cf_url = m.group(0)
    if not lt_url and os.path.exists(LT_LOG):
        with open(LT_LOG) as f:
            m = re.search(r"https://[a-zA-Z0-9-]+\.loca\.lt", f.read())
            if m:
                lt_url = m.group(0)
    if (not BAT_CLOUDFLARE or cf_url) and (not BAT_LOCALTUNNEL or lt_url):
        break

# 10. In tất cả link truy cập
print("\n" + "═"*65)
print("🔗 ĐƯỜNG DẪN TRUY CẬP COMFYUI")
print("─"*65)
if google_proxy_url:
    print(f"🌟 [KHUYẾN NGHỊ] Google Colab (100% không chặn):")
    print(f"   {google_proxy_url}")
if cf_url:
    print(f"🌐 Cloudflare Tunnel (dự phòng 1): {cf_url}")
if lt_url:
    print(f"🔄 Localtunnel (dự phòng 2):       {lt_url}")
    print(f"   (Mật khẩu Localtunnel: {colab_ip})")
if not google_proxy_url and not cf_url and not lt_url:
    print("⚠️  Không lấy được link! Kiểm tra lại Bước 3 và thử chạy lại cell này.")
print("─"*65)
print(f"📁 Ảnh sinh ra lưu tại: Google Drive → ComfyUI_Outputs")
print("═"*65 + "\n")

# 11. Live stream log ComfyUI
print("📋 STREAM LOG COMFYUI (Đang hoạt động... Ctrl+C để dừng):")
try:
    with open(COMFY_LOG, "r") as f:
        f.seek(0, 2)
        while True:
            line = f.readline()
            if line:
                print(line, end="")
            else:
                time.sleep(0.5)
except KeyboardInterrupt:
    print("\n🛑 Đã dừng stream log. ComfyUI vẫn đang chạy nền.")

In [ ]:
# @title 🔄 (Tùy Chọn) Khởi Động Lại ComfyUI Không Mất Dữ Liệu
# @markdown Chạy cell này khi ComfyUI bị treo hoặc muốn restart sau khi cài thêm node mới.
import subprocess, os, time, re, urllib.request

COMFY_DIR = "/content/ComfyUI"
COMFY_LOG = "/content/comfyui.log"

print("🔄 Đang dừng tiến trình ComfyUI cũ...")
!pkill -f "python main.py" 2>/dev/null || true
!fuser -k 8188/tcp 2>/dev/null || true
time.sleep(2)

if os.path.exists(COMFY_LOG):
    os.remove(COMFY_LOG)

%cd "{COMFY_DIR}"
comfy_cmd = f"python main.py --listen 0.0.0.0 --port 8188 --enable-cors-header > {COMFY_LOG} 2>&1"
subprocess.Popen(comfy_cmd, shell=True)

print("⏳ Đang khởi động lại ComfyUI...")
comfy_ready = False
for i in range(60):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/", timeout=2) as resp:
            if resp.status == 200:
                comfy_ready = True
                break
    except:
        pass
    time.sleep(2)

if comfy_ready:
    print("\n✅ ComfyUI đã khởi động lại thành công!")
    print("   → Làm mới trang ComfyUI trên trình duyệt để tiếp tục.")
else:
    print("\n⚠️  ComfyUI chưa sẵn sàng. Xem log:")
    if os.path.exists(COMFY_LOG):
        with open(COMFY_LOG) as f:
            print(f.read()[-2000:])

In [ ]:
# @title 📊 (Tùy Chọn) Xem Báo Cáo Tổng Kết Cài Đặt
# @markdown Hiển thị báo cáo chi tiết: Models tải thành công, tải thất bại, thời gian.
import os

COMFY_DIR   = "/content/ComfyUI"
REPORT_FILE = f"{COMFY_DIR}/SETUP_REPORT.md"

if os.path.exists(REPORT_FILE):
    from IPython.display import Markdown, display
    with open(REPORT_FILE) as f:
        content = f.read()
    display(Markdown(content))
else:
    print("ℹ️  Chưa có báo cáo. Hãy chạy Bước 3 (cài đặt) trước.")
    print(f"   Đường dẫn báo cáo sẽ là: {REPORT_FILE}")